# 강의 03 · 실습 4 — RAG 에이전트 서비스 · (4) 고난도 I

## 1. 문제상황

- 구름월드 안내 서비스는 임계값을 통과한 청크(chunk)면 그대로 근거로 씁니다.
- 그런데 「겨울에 눈썰매장 운영하나요」처럼 임계값은 통과했지만 청크가 질문에 못 미치는 경우가 있습니다. 점수는 가깝지만 답이 아닌 청크입니다.
- 지금은 이런 청크로 모델이 답을 만들어, 질문과 어긋난 답이 나갑니다.
- 담당자는 검색 결과가 질문의 근거로 충분한지 먼저 판정하고, 부족하면 질의를 고쳐 다시 검색하되 상한 안에서만 되풀이하고, 상한에 닿으면 고정 안내 문장으로 끝내기를 원합니다.

## 2. 문제와 목표

- **문제**: 임계값 통과와 「질문의 근거로 충분함」은 다른데, 지금은 통과한 청크를 그대로 근거로 씁니다.
- **목표**: 검색 결과를 두 값(yes/no) 중 하나로 판정하는 구조화 출력 판정기를 두고, no이면 앞 질의와 그 질의가 데려온 근거를 보여 주며 다른 각도로 질의를 다시 써 재검색하고, 재시도 상한(2회)에 닿으면 고정 안내 문장으로 끝내는 처리 함수를 만들어 `POST /ask`로 노출합니다.
    - 판정기: 구조화 출력 `Grade`(sufficient: yes/no) — 검색 결과가 질문의 근거로 충분한지.
    - 질의 재작성: 원 질문, 앞 질의, 앞 질의가 데려온 근거를 주고 다른 각도의 한 문장으로.
    - 재시도 상한: 2회. 닿으면 고정 안내 문장 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」
    - 처리 함수: corrective_answer.
- **목표 달성 여부의 판정 기준**: 환불 질문은 첫 판정이 충분함으로 나와 근거로 답하고, 「겨울에 눈썰매장 운영하나요?」는 회차마다 다른 질의로 판정이 부족함으로 이어져 상한에 닿은 뒤 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」로 끝나며, 서비스 호출에서도 같은 답이 돌아오는 것을 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex04_s4_diagram.svg)

## 4. 단계별 요구사항

1. **문서를 적재합니다.**
    - `day05_faq_구름월드.csv`(32행)를 `csv.DictReader`로 읽고(`utf-8-sig`), `Question`이 빈 행은 버립니다.
    - 행마다 `[카테고리] Q: 질문\nA: 답변` 형식의 본문과 `{"row": 행 번호, "category": 카테고리}` 메타데이터를 가진 `Document`를 만들어 리스트 `docs`에 모으고, 적재 문서 수를 출력합니다.
2. **임베딩을 준비합니다.**
    - `OpenAIEmbeddings(model="text-embedding-3-small")`로 임베딩 부품 `emb`를 만듭니다.
3. **저장소를 구축하고 영속합니다.**
    - `Chroma.from_documents(docs, emb, persist_directory="chroma_db", ids=[...])`로 저장소 `db`를 만들고, 문서 id는 `row-<행 번호>`로 주어 셀을 다시 실행해도 항목이 늘지 않게 합니다.
4. **점수와 함께 검색합니다.**
    - `retrieve(query, k=2)`로 `db.similarity_search_with_score`의 청크·점수 목록을 돌려줍니다.
5. **판정기를 선언합니다.**
    - `Grade(BaseModel)`에 `sufficient: Literal["yes", "no"]` 한 칸을 두고, `grader = llm.with_structured_output(Grade)`로 판정기를 만듭니다.
    - 「자유이용권 환불 규정」의 검색 결과로 한 번 시험해 `yes`를 확인합니다.
6. **입출력 모양과 고정 문장을 선언합니다.**
    - `AskIn(question: str)`, `AskOut(answer: str)`, 고정 안내 문장 `NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."`를 둡니다.
7. **처리 함수를 만듭니다.**
    - `corrective_answer(question, max_retry=2)`는 회차마다 `retrieve(query)`로 청크를 모아 근거 문자열을 만들고, 판정기에 「질문·검색 결과·이 검색 결과가 질문에 답할 근거로 충분한가」를 넣어 `yes`/`no`를 받습니다.
    - `yes`면 「아래 근거로만 답하라」 프롬프트로 답을 만들어 돌려주고, `no`면(상한 안이면) 원 질문·앞 질의·앞 질의가 데려온 근거 앞부분을 주고 「다른 각도로 한 문장으로 바꿔라. 바꾼 질문만 출력하라」로 질의를 다시 씁니다.
    - 회차·질의·판정을 출력하고, 상한에 닿으면 `NO_EVIDENCE`를 돌려줍니다.
    - 「자유이용권 환불 규정 알려 주세요」와 「겨울에 눈썰매장 운영하나요?」로 시험합니다.
8. **앱과 엔드포인트를 등록합니다.**
    - `%%writefile app_s4.py`로 서비스 파일을 만듭니다.
    - 파일은 `.env`를 읽고 `chroma_db`를 다시 열어 판정기·처리 함수를 그대로 담고, `app = FastAPI()`, `GET /healthz`, `POST /ask`(`AskIn`을 받아 `AskOut(answer=corrective_answer(...))`를 돌려줌)를 등록합니다.
9. **기동하고 호출을 확인합니다.**
    - `fastapi dev app_s4.py --port 8031 --no-reload`를 `subprocess`로 띄우고 `/healthz`가 200을 줄 때까지 기다린 뒤, `samples_crag.json`의 질문 두 개를 `httpx.post("/ask")`로 보내 상태 코드와 응답 JSON을 출력하고, 서버를 종료합니다.
    - 처리 함수는 회차마다 「[판정 N회차] query=질의 → yes/no」 줄을, 서버 확인은 「[healthz] 200 ok」 줄을 출력합니다.
    - 서비스 주소는 `http://127.0.0.1:8031`입니다.

## 5. 코드 골격 — RAG 인덱싱·검색 5단 + FastAPI 서비스 4단

두 골격을 잇습니다. ⑤가 판정기이고, 서비스 ②의 처리 함수가 교정 재검색 루프입니다.

**RAG 인덱싱·검색 5단** (검색 도구 안에 접힙니다)

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 문서 적재 | 원본 파일을 읽어 검색 단위 문서로 만듭니다 | `Document(page_content=..., metadata=...)` | 1 |
| ② 임베딩 준비 | 문장을 숫자 벡터로 바꿀 모델을 지정합니다 | `OpenAIEmbeddings(model="text-embedding-3-small")` | 2 |
| ③ 저장소 구축·영속 | 문서와 임베딩을 넣어 저장소를 만들고 디렉터리에 남깁니다 | `Chroma.from_documents(docs, emb, persist_directory=...)` | 3 |
| ④ 점수 동반 검색 | 질문을 넣어 가까운 문서와 그 거리 점수를 함께 받습니다 | `db.similarity_search_with_score(q, k=2)` | 4 |
| ⑤ 임계값 컷 | 점수가 기준을 넘으면 문서 근거를 쓰지 않고 다른 경로로 보냅니다 | `class Grade(BaseModel)`, `llm.with_structured_output(Grade)` | 5 |

**FastAPI 서비스 4단** (서비스 표준 템플릿 `service_template`의 구성과 같습니다)

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| 서비스 ① 입출력 모양 선언 | 서비스가 받을 값과 돌려줄 값의 모양을 클래스로 선언합니다 | `class AskIn(BaseModel)`, `class AskOut(BaseModel)` | 6 |
| 서비스 ② 처리 함수 구현 | 값을 받아 결과를 돌려주는 함수를 웹과 무관하게 먼저 만듭니다 | `def corrective_answer(question, max_retry=2) -> str` | 7 |
| 서비스 ③ 앱·엔드포인트 등록 | 앱을 만들고, 주소와 함수를 데코레이터로 잇습니다 | `app = FastAPI()`, `@app.post("/ask")` | 8 |
| 서비스 ④ 기동·호출 확인 | 개발 서버를 띄우고 요청을 보내 응답을 확인합니다 | `fastapi dev app_s4.py --port ...`, `httpx.post("/ask")` | 9 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 읽습니다.

- API 키는 `.env` 파일에서 읽습니다. 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, `OPENAI_API_KEY=발급받은_키` 한 줄만 넣습니다.
- 서비스 파일 `app_s4.py`도 같은 `.env`를 `find_dotenv(usecwd=True)`로 읽습니다.

In [ ]:
import csv
import json
import os
import subprocess
import sys
import time

import httpx
from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from typing import Literal

from pydantic import BaseModel

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

### 단계 ① — 문서 적재 (요구사항 1)

- FAQ 한 행을 청크 하나로 삼고, 질문과 답을 한 본문에 넣습니다.

In [ ]:
# 여기에 단계 ①(문서 적재)을 작성합니다.

### 단계 ② — 임베딩 준비 (요구사항 2)

- 문장을 숫자 벡터로 바꿀 임베딩 모델을 지정합니다.

In [ ]:
# 여기에 단계 ②(임베딩 준비)를 작성합니다.

### 단계 ③ — 저장소 구축·영속 (요구사항 3)

- 저장소를 디렉터리에 남깁니다. 서비스 파일 `app_s4.py`는 이 디렉터리를 다시 열어 씁니다.

In [ ]:
# 여기에 단계 ③(저장소 구축·영속)을 작성합니다.

### 단계 ④ — 점수 동반 검색 (요구사항 4)

- 점수 동반 검색을 함수 `retrieve`로 감쌉니다. 도구는 이 함수를 부릅니다.

In [ ]:
# 여기에 단계 ④(retrieve 함수)를 작성합니다.

### 단계 ⑤ — 판정기 선언 (요구사항 5)

- 임계값 컷 대신 「근거로 충분한가」를 모델이 판정합니다. 두 값 중 하나만 담는 규격으로 받아 분기가 값 하나를 보고 나뉘게 합니다.

In [ ]:
# 여기에 단계 ⑤(Grade 스키마와 판정기, 시험 판정)를 작성합니다.

### 서비스 단계 ① — 입출력 모양 선언 (요구사항 6)

- 요청 본문과 응답 본문의 모양을 pydantic 클래스로 못 박습니다. 고정 안내 문장도 여기서 선언합니다.

In [ ]:
# 여기에 서비스 단계 ①(AskIn·AskOut·NO_EVIDENCE)을 작성합니다.

### 서비스 단계 ② — 처리 함수 구현 (요구사항 7)

- 회차마다 검색 → 판정 → (yes) 답 생성 / (no) 질의 재작성의 순서로 돕니다.
- 재작성은 원 질문만 보면 회차마다 같은 문장이 나옵니다. 앞 질의와 그 질의가 데려온(부족했던) 근거를 함께 주고 다른 각도를 요구합니다.

In [ ]:
# 여기에 서비스 단계 ②(corrective_answer와 두 질문 시험)를 작성합니다.

### 서비스 단계 ③ — 앱·엔드포인트 등록 (요구사항 8)

- 서비스 파일은 노트북과 따로 도는 프로그램이므로 저장소를 다시 열고 도구·처리 함수를 파일 안에 그대로 둡니다.
- `@app.post("/ask")`가 주소와 함수를 잇습니다. 함수 안에서 처리 함수를 그대로 부르고, 반환한 값이 JSON 응답 본문이 됩니다.

In [ ]:
%%writefile app_s4.py
# 여기에 서비스 단계 ③(app_s4.py 전체)을 작성합니다.

### 서비스 단계 ④ — 기동·호출 확인 (요구사항 9)

- 노트북에서 개발 서버를 자식 프로세스로 띄우고, 살아 있는지 확인한 뒤 요청을 보내고, 끝나면 서버를 내립니다.
- 터미널에서는 `fastapi dev app_s4.py --port 8031`로 띄우고 다른 터미널에서 `python run_samples.py 8031`로 같은 확인을 합니다. PowerShell의 `curl`은 별칭이므로 `curl.exe`나 `run_samples.py`를 씁니다.
- 포트 8031이 이미 쓰이고 있으면 노트북과 명령의 포트 숫자를 함께 바꿉니다.

In [ ]:
# 여기에 서비스 단계 ④(서버 기동, healthz 대기, samples.json 호출, 종료)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 단계 ⑤의 시험 판정이 `yes`입니다.
2. 서비스 단계 ②에서 환불 질문은 `[판정 1회차] … yes` 뒤 근거로 답하고, 눈썰매장 질문은 `[판정 1회차]`·`[판정 2회차]`·`[판정 3회차]`의 질의가 서로 다르며 모두 `no`인 뒤 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」로 끝납니다.
3. 서비스 단계 ④에서 두 요청의 상태가 200이고 응답 `answer`가 단계 ②와 같은 성격입니다.

세 가지가 모두 확인되면 완성입니다. 재작성된 질의의 문장은 실행마다 달라집니다.